# Cell-type Classification — waveform + spike-train + WaveMAP

**Classification only** (no WM analysis). Produces a per-unit **label** table for the
Sternberg working-memory datasets, in the style of `neural_01c`. Three schemes:

1. **narrow / broad** — trough-to-peak waveform width (GMM antimode split)
2. **interneuron verification** — is the narrow group fast-spiking & inhibitory? (ISI/ACG test)
3. **WaveMAP** — data-driven waveform clusters (UMAP graph → Louvain)

Engine lives in the `celltyping` package; every step here is the transparent version of
`celltyping.labels.build_label_table()`. NOTE: 000469 has no stored waveforms, so its units
get spike-train features only (`wf_group = NA`, WaveMAP skipped).

## Cell index & knobs

`DATASET` / `DATA_ROOT` → `[0.1]` · file selection → `[0.2]` · QC gates → `[0.2b]` ·
split method → `[2.0]` · WaveMAP params → `[4.0]`

Sections: `[0.x]` setup · `[1.x]` features + QC · `[2.x]` narrow/broad split ·
`[3.x]` interneuron verification · `[4.x]` WaveMAP · `[5.x]` label table

## Section 0: Setup & data

In [ ]:
# [0.1] imports & config
import sys, os, warnings, time
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = Path.cwd()                      # run this notebook from E:\SBCAT\celltyping
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from celltyping.features import build_unit_features, autocorrelogram, OSORT_FS_HZ
from celltyping.classify import assign_narrow_broad, interneuron_verification
from celltyping.wavemap import (preprocess_waveforms, run_wavemap,
                                characterize_clusters, cluster_waveform_importance)
from celltyping import viz

plt.rcParams['figure.dpi'] = 100
sns.set_palette('Set2')
NB_COLORS = {'narrow': '#d62728', 'broad': '#1f77b4'}
RS = 42

# ---- DATA KNOBS ----
DATASET   = '000673'                       # '000673' (waveforms) | '000469' (no waveforms)
DATA_ROOT = PROJECT.parent / 'data' / DATASET
FS_HZ     = 100_000.0                      # OSort convention; ttp ~0.2-1.0 ms confirms

all_files = sorted(DATA_ROOT.glob('sub-*/*.nwb'))
print(f'{DATASET}: {len(all_files)} NWB files under {DATA_ROOT}')
for i, f in enumerate(all_files[:6]):
    print(f'  [{i}] {f.parent.name}/{f.name}')
if len(all_files) > 6:
    print(f'  ... (+{len(all_files)-6} more)')

In [ ]:
# [0.2] choose files  (None/[] = all)
SELECTED_INDICES = []          # e.g. [0,1,2] to prototype on a few sessions
SELECT_SUBSTRING = None        # e.g. 'sub-1_'

if SELECTED_INDICES:
    files = [all_files[i] for i in SELECTED_INDICES]
elif SELECT_SUBSTRING:
    files = [f for f in all_files if SELECT_SUBSTRING in f.name]
else:
    files = list(all_files)
assert files, 'no files selected'
print(f'using {len(files)} / {len(all_files)} files')

In [ ]:
# [0.2b] QC gate knobs (used for qc_pass + WaveMAP inclusion)
MIN_PEAK_SNR = None            # e.g. 3.0 to match WaveMAP's SNR>=3 (None = off)
MAX_ISI_VIOL = 0.05            # max fraction ISI<2ms (contamination). WaveMAP uses 0.0025
print(f'QC gates: peak_snr>={MIN_PEAK_SNR}, prop_isi_viol<={MAX_ISI_VIOL}')

## Section 1: Per-unit features (waveform + spike-train/ACG)

In [ ]:
# [1.1] extract features  (returns waveforms for WaveMAP too)
t0 = time.time()
df, wfs = build_unit_features(files, fs_hz=FS_HZ, return_waveforms=True, verbose=True)
print(f'\n{len(df)} units in {time.time()-t0:.0f}s | subjects={df.subject.nunique()} sessions={df.session_id.nunique()}')
df.head()

In [ ]:
# [1.1b] compute qc_pass exactly as build_label_table does
qc = df['wf_valid'].fillna(False) & df['st_valid'].fillna(False)
if MAX_ISI_VIOL is not None:
    qc &= pd.to_numeric(df['prop_isi_viol'], errors='coerce').fillna(1.0) <= MAX_ISI_VIOL
if MIN_PEAK_SNR is not None and 'waveforms_peak_snr' in df:
    qc &= pd.to_numeric(df['waveforms_peak_snr'], errors='coerce').fillna(0.0) >= MIN_PEAK_SNR
df['qc_pass'] = qc.to_numpy()
print(f'qc_pass: {int(df.qc_pass.sum())}/{len(df)} units | '
      f'valid waveforms: {int(df.wf_valid.sum())} | valid spike-trains: {int(df.st_valid.sum())}')

In [ ]:
# [1.2] QC feature distributions
qc_feats = [('waveforms_peak_snr','peak SNR'), ('waveforms_isolation_distance','isolation dist'),
            ('prop_isi_viol','ISI viol frac'), ('mean_rate_hz','firing rate (Hz)'),
            ('n_spikes','# spikes'), ('half_width_ms','half-width (ms)')]
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, (c, lab) in zip(axes.ravel(), qc_feats):
    if c not in df: ax.set_visible(False); continue
    s = pd.to_numeric(df[c], errors='coerce').dropna()
    if c in ('waveforms_isolation_distance','n_spikes'):
        s = s[s>0]; ax.set_xscale('log')
    ax.hist(s, bins=40, color='#4c9', alpha=0.8)
    ax.set_title(lab, fontsize=9); ax.set_ylabel('# units', fontsize=8)
fig.suptitle(f'[{DATASET}] QC feature distributions ({len(df)} units)', fontweight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# [1.3] cell-typing feature distributions
feat_list = [('trough_to_peak_ms','trough-to-peak (ms)'), ('half_width_ms','half-width (ms)'),
             ('mean_rate_hz','rate (Hz)'), ('isi_cv','ISI CV'),
             ('burst_index','burst index'), ('acg_refrac_ms','ACG refractory (ms)')]
v = df[df.wf_valid & df.st_valid]
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, (c, lab) in zip(axes.ravel(), feat_list):
    s = pd.to_numeric(v[c], errors='coerce').dropna()
    ax.hist(s, bins=40, color='#888', alpha=0.8); ax.set_title(lab, fontsize=9)
fig.suptitle(f'[{DATASET}] cell-typing features (valid units, n={len(v)})', fontweight='bold')
fig.tight_layout(); plt.show()

## Section 2: Narrow vs broad waveform split

In [ ]:
# [2.0] split by trough-to-peak
SPLIT_METHOD = 'antimode'      # 'antimode' (GMM valley) | 'median' | 'fixed'
SPLIT_FIXED_MS = 0.5           # used only if SPLIT_METHOD=='fixed'
grp, SPLIT_MS = assign_narrow_broad(df, method=SPLIT_METHOD, fixed_ms=SPLIT_FIXED_MS)
df['wf_group'] = grp
vc = df.loc[df.wf_valid, 'wf_group'].value_counts().to_dict()
n_nb = sum(vc.get(k,0) for k in ('narrow','broad'))
print(f'split = {SPLIT_MS:.3f} ms | {vc} | '
      f"{100*vc.get('narrow',0)/max(n_nb,1):.0f}% narrow")

In [ ]:
# [2.1] width histogram (by group) + ttp-vs-fwhm
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
viz.plot_waveform_split(df, split_ms=SPLIT_MS, ax=axes[0])
v = df[df.wf_valid]
for g in ('broad','narrow'):
    s = v[v.wf_group==g]
    axes[1].scatter(s['trough_to_peak_ms'], s['half_width_ms'], s=8, alpha=0.4,
                    color=NB_COLORS[g], label=g)
axes[1].axvline(SPLIT_MS, color='k', ls='--', lw=1)
axes[1].set_xlabel('trough-to-peak (ms)'); axes[1].set_ylabel('half-width (ms)')
axes[1].set_title('width metrics'); axes[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

In [ ]:
# [2.2] session-confound QC: is narrow/broad a WITHIN-session split? (cf. neural_01c [1.1c])
from sklearn.mixture import GaussianMixture
v = df[df.wf_valid].copy()
rows = []
for sid, s in v.groupby('session_id'):
    x = s['trough_to_peak_ms'].dropna().values.reshape(-1,1); n = len(x)
    b1 = GaussianMixture(1, random_state=0).fit(x).bic(x)
    b2 = GaussianMixture(2, random_state=0).fit(x).bic(x) if n>=10 else np.nan
    rows.append(dict(session=sid.split('_ses')[0].replace('sub-',''), n=n,
                     med_ttp=round(float(np.median(x)),3),
                     pct_narrow=round(100*(s.wf_group=='narrow').mean()),
                     within_bimodal='yes' if (np.isfinite(b2) and b2<b1-2) else 'no'))
sess_tab = pd.DataFrame(rows).sort_values('session')
ov = 100*(v.wf_group=='narrow').mean()
print(f'narrow fraction overall {ov:.0f}% | range {sess_tab.pct_narrow.min():.0f}-{sess_tab.pct_narrow.max():.0f}% | '
      f"within-session bimodal: {(sess_tab.within_bimodal=='yes').sum()}/{len(sess_tab)}")
fig, ax = plt.subplots(figsize=(12,3.6))
ax.bar(range(len(sess_tab)), sess_tab.pct_narrow, color='#d62728', alpha=0.8)
ax.axhline(ov, color='k', ls='--', lw=1, label=f'overall {ov:.0f}%')
ax.set_xticks(range(len(sess_tab))); ax.set_xticklabels(sess_tab.session, rotation=90, fontsize=6)
ax.set_ylabel('% narrow'); ax.set_title('Narrow fraction per session'); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()
sess_tab

In [ ]:
# [2.3] mean waveform per group (polarity-normalized, trough-aligned)
PRE_MS, POST_MS = 0.6, 1.2
wf_present = np.array([w is not None for w in wfs])
Xall, valid_all = preprocess_waveforms([w for w in wfs], fs_hz=FS_HZ, pre_ms=PRE_MS, post_ms=POST_MS)
groups_all = df.loc[wf_present.nonzero()[0] if False else np.where(valid_all)[0], 'wf_group'].to_numpy()              if valid_all.any() else np.array([])
# align groups to the valid rows of Xall
grp_valid = df['wf_group'].to_numpy()[np.where(valid_all)[0]]
fig, ax = plt.subplots(figsize=(6,4))
viz.plot_mean_waveforms(Xall, grp_valid, palette=NB_COLORS, ax=ax)
ax.set_title(f'[{DATASET}] mean waveform by group'); plt.show()

## Section 3: Is the narrow group fast-spiking inhibitory interneurons?

In [ ]:
# [3.1] interneuron verification stats (narrow vs broad, one-sided MWU in predicted direction)
ver = interneuron_verification(df, group_col='wf_group')
print(ver.to_string(index=False, float_format=lambda x: f'{x:.4g}'))
print(f"\n=> {int(ver.supports.sum())}/{len(ver)} predictions supported at p<0.05")

In [ ]:
# [3.2] verification boxplots
fig = viz.plot_interneuron_verification(df, group_col='wf_group'); plt.show()

In [ ]:
# [3.3] example autocorrelograms: narrow (red) vs broad (blue)
import numpy as np
from pynwb import NWBHDF5IO
def _acg_for_unit(uid):
    sid = uid.rsplit('_u',1)[0]; ui = int(uid.rsplit('_u',1)[1])
    fp = next(f for f in files if f.stem==sid)
    io = NWBHDF5IO(str(fp),'r',load_namespaces=True); nwb=io.read()
    st = np.asarray(nwb.units['spike_times'][ui][:], dtype=float); io.close()
    return autocorrelogram(st, bin_ms=1.0, win_ms=50.0)
rng = np.random.default_rng(RS)
fig, axes = plt.subplots(2, 4, figsize=(14, 5.5))
for r, g in enumerate(('narrow','broad')):
    pool = df[(df.wf_group==g) & df.qc_pass].sort_values('mean_rate_hz', ascending=False)
    pick = pool.unit_id.head(20).sample(min(4,len(pool)), random_state=RS).tolist()
    for ax, uid in zip(axes[r], pick):
        cnt, ctr = _acg_for_unit(uid)
        ax.bar(ctr, cnt, width=1.0, color=NB_COLORS[g], alpha=0.85)
        ax.set_title(f'{g}\n{uid.split("_ses")[0].replace("sub-","")}', fontsize=7)
        ax.set_xlabel('lag (ms)', fontsize=7)
fig.suptitle('Example autocorrelograms', fontweight='bold'); fig.tight_layout(); plt.show()

## Section 4: WaveMAP — data-driven waveform clusters

Runs on QC-passing units with valid waveforms. `resolution` up → fewer clusters.
(000469 has no waveforms → this section will find nothing to cluster.)

In [ ]:
# [4.0] WaveMAP knobs + run
N_NEIGHBORS = 15
RESOLUTION  = 2.0
use_mask = df['qc_pass'].to_numpy() & np.array([w is not None for w in wfs])
use_idx  = np.where(use_mask)[0]
print(f'{use_idx.size} units into WaveMAP (>=300 ideal)')
wm = None
if use_idx.size >= 20:
    X, valid = preprocess_waveforms([wfs[i] for i in use_idx], fs_hz=FS_HZ, pre_ms=PRE_MS, post_ms=POST_MS)
    wm = run_wavemap(X, n_neighbors=N_NEIGHBORS, resolution=RESOLUTION, random_state=RS)
    kept = use_idx[valid]
    df['wavemap_cluster'] = pd.array([pd.NA]*len(df), dtype='Int64')
    df.loc[df.index[kept], 'wavemap_cluster'] = wm['labels']
    print(f"clusters: {np.bincount(wm['labels']).tolist()}")
else:
    df['wavemap_cluster'] = pd.array([pd.NA]*len(df), dtype='Int64')
    print('skipped — too few units (no waveforms?)')

In [ ]:
# [4.1] UMAP embedding, colored 3 ways
if wm is not None:
    sub = df.iloc[use_idx[valid]].reset_index(drop=True)   # feat rows aligned to X / labels
    fig = viz.plot_wavemap_embedding(wm['embedding'], feat_df=sub, labels=wm['labels']); plt.show()
else:
    print('no WaveMAP result')

In [ ]:
# [4.2] mean waveform per WaveMAP cluster
if wm is not None:
    fig, ax = plt.subplots(figsize=(6,4))
    viz.plot_mean_waveforms(wm['X'], wm['labels'], ax=ax)
    ax.set_title(f'[{DATASET}] mean waveform per WaveMAP cluster'); plt.show()

In [ ]:
# [4.3] cluster physiology profile + cross-tabs
if wm is not None:
    print('per-cluster median physiology:')
    print(characterize_clusters(df, cluster_col='wavemap_cluster').to_string())
    print('\ncluster x narrow/broad:')
    print(pd.crosstab(df.wavemap_cluster, df.wf_group).to_string())
    print('\ncluster x region:')
    print(pd.crosstab(df.wavemap_cluster, df.location).to_string())

In [ ]:
# [4.4] (optional, slower) which waveform samples define the clusters — XGBoost + SHAP
RUN_SHAP = False
if wm is not None and RUN_SHAP:
    imp = cluster_waveform_importance(wm['X'], wm['labels'], random_state=RS)
    print(f"cross-val accuracy waveform->cluster: {imp['accuracy']:.2f}")
    fig, ax = plt.subplots(figsize=(8,3))
    ax.plot(imp['mean_abs_shap'], color='#a33'); ax.set_xlabel('waveform sample')
    ax.set_ylabel('mean |SHAP|'); ax.set_title('sample importance for cluster identity'); plt.show()

## Section 5: Per-unit label table (the deliverable)

In [ ]:
# [5.1] assemble + save the label table  (== celltyping.labels.build_label_table one-liner)
from celltyping.labels import _PUTATIVE, LABEL_COLS, save_label_table
df['putative_type'] = df['wf_group'].map(_PUTATIVE)
df.insert(1, 'dataset', DATASET)
keep = ['dataset'] + [c for c in LABEL_COLS if c in df.columns]
label_df = df[keep].copy()

out = PROJECT / 'outputs' / 'celltype' / f'unit_labels_{DATASET}.csv'
save_label_table(label_df, out)
print(f'wrote {out}  ({len(label_df)} units)')
print('\nlabel counts:')
print('  wf_group      :', df.wf_group.value_counts(dropna=False).to_dict())
print('  putative_type :', df.putative_type.value_counts(dropna=False).to_dict())
print('  wavemap_cluster:', df.wavemap_cluster.value_counts(dropna=False).to_dict())
label_df.head(12)

---
*One-call equivalent of this whole notebook:*
```python
from celltyping.labels import build_label_table, save_label_table
label_df, extras = build_label_table(files, fs_hz=FS_HZ, dataset=DATASET, do_wavemap=True)
save_label_table(label_df, out)
```